# DropGuard — Exploratory Data Analysis

This notebook explores the UCI Student Performance dataset used by the DropGuard early-warning system.
We examine label distributions, feature correlations, and key patterns that inform model design.

In [ ]:
import sys
import yaml
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Resolve project root and add src/ to path so dropguard imports work
PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dropguard.data import load_config, load_raw_data, create_labels, clean_data

config = load_config(PROJECT_ROOT / 'config.yaml')
print('Config loaded:', list(config.keys()))

df_raw = load_raw_data(config)
df = create_labels(df_raw, config)
df = clean_data(df, config)
print(f'Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns')

In [ ]:
print('Shape:', df.shape)
print()
print('Column dtypes:')
print(df.dtypes)
print()
df.head()

In [ ]:
df.describe()

In [ ]:
label_counts = df['at_risk'].value_counts().rename({0: 'Not At-Risk', 1: 'At-Risk'})
print('Label distribution:')
print(label_counts)
print(f'\nAt-risk rate: {label_counts["At-Risk"] / len(df):.1%}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
label_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white', rot=0)
axes[0].set_title('At-Risk Label Distribution')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width() / 2, p.get_height() + 5),
                     ha='center', fontsize=11)

label_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                  colors=['steelblue', 'tomato'], startangle=140)
axes[1].set_title('Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'figures' / 'label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.4, square=True, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Numeric Features', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'figures' / 'correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
grade_cols = ['G1', 'G2', 'G3']
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)

for ax, col in zip(axes, grade_cols):
    df.boxplot(column=col, by='at_risk', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='tomato', linewidth=2))
    ax.set_title(f'{col} by At-Risk Label')
    ax.set_xlabel('at_risk (0=No, 1=Yes)')
    ax.set_ylabel('Grade')

plt.suptitle('Grade Distributions by At-Risk Label', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'figures' / 'grade_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column='absences', by='at_risk', ax=ax,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='tomato', linewidth=2),
           flierprops=dict(marker='o', alpha=0.4))
ax.set_title('Absences Distribution by At-Risk Label')
ax.set_xlabel('at_risk (0=Not At-Risk, 1=At-Risk)')
ax.set_ylabel('Number of Absences')
plt.suptitle('')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'figures' / 'absences_by_risk.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(
    data=df, x='failures', hue='at_risk',
    palette={0: 'steelblue', 1: 'tomato'}, ax=ax, edgecolor='white'
)
ax.set_title('Past Course Failures by At-Risk Label')
ax.set_xlabel('Number of Past Failures')
ax.set_ylabel('Count')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, ['Not At-Risk', 'At-Risk'], title='at_risk')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'figures' / 'failures_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## Key Observations

- **Class imbalance is moderate**: roughly 30–40 % of students are labelled at-risk (G3 < 10), which calls for stratified splits and imbalance-aware metrics (F1, ROC-AUC) rather than raw accuracy.
- **Grade features are strongly predictive**: G1 and G2 correlate highly with the at-risk label and with each other (r ≈ 0.85–0.90), confirming that early-term performance is the single strongest signal for dropout risk.
- **Past failures compound risk**: students with one or more past course failures are disproportionately represented in the at-risk group, suggesting that historical academic setbacks should be a prominent feature.
- **Absences show a right-skewed distribution with elevated medians for at-risk students**: while the overlap is large, high absenteeism (> 10 days) is a secondary behavioural indicator worth including alongside grade features.